In [35]:
import os
import time
import pandas as pd
from ytmusicapi import YTMusic

PLAYLIST_DIR = '../playlists/'
AUTH = '../headers_auth.json'
TRACKS_DB_TSV = os.path.join(PLAYLIST_DIR, '_tracks_db.tsv')
LIKE_TRACKS_TSV = os.path.join(PLAYLIST_DIR, '_liked_tracks.tsv')
LIKE_TRACKS_HEADER = ['title', 'album', 'artist'] # more compact

yt_api = YTMusic(AUTH)



In [63]:
%%time

def update_like_tsv(liked_tracks, like_tsv=LIKE_TRACKS_TSV, header=LIKE_TRACKS_HEADER):
    # Load already existing like list tsv
    like_tracks = pd.read_csv(like_tsv, sep='\t', index_col=0)
    assert list(like_tracks.columns) == header, 'Expected %s to have header %s, not: %s' % (
        like_tsv, header, like_tracks.columns)

    # Append new like tracks in db but not in like list, save tsv.
    new_like_tracks = liked_tracks.loc[set(liked_tracks.index) - set(like_tracks.index)]
    all_like_tracks = pd.concat([like_tracks,new_like_tracks])
    all_like_tracks.to_csv(like_tsv, sep='\t', header=True)
    print('Updated liked tracks with %d new entries growing it from %d to %d entries.' % (
        len(new_like_tracks), len(like_tracks), len(all_like_tracks)))
    return all_like_tracks

# Load current db and filter out like tracks
track_db = pd.read_csv(TRACKS_DB_TSV, sep='\t', index_col=0)
track_db_liked = track_db.loc[track_db['likeStatus'] == 'LIKE', LIKE_TRACKS_HEADER]
new_like_df = update_like_tsv(track_db_liked)
new_like_df.describe()



Updated liked tracks with 0 new entries growing it from 17840 to 17840 entries.
CPU times: user 401 ms, sys: 53.6 ms, total: 455 ms
Wall time: 474 ms


,title,album,artist
count,17840,17713,17818
unique,15870,7098,3812
top,Intro,Greatest Hits,The Beatles
freq,17,39,188


In [64]:
def is_like_pl(name):
    name = name.lower()
    if 'thumbs_up' in name:
        return True
    if ' like' in name or ' likes' in name:
        return True
    if ' top' in name:
        return True

playlist_files = sorted(os.listdir(PLAYLIST_DIR))
like_playlists = [pl for pl in playlist_files if is_like_pl(pl)]
print('Found %d like playlists out ot the %d total' % (len(like_playlists), len(playlist_files)))

# Concat liked playlist tracks
playlist_liked_tracks = []
for pl in like_playlists:
    track_df = pd.read_csv(os.path.join(PLAYLIST_DIR, pl), sep='\t', index_col=0)
    tracks_db_liked = track_df.loc[track_df['likeStatus'] == 'LIKE']
    tracks_db_liked = tracks_db_liked.set_index('videoId', drop=True)
    playlist_liked_tracks.append(tracks_db_liked)
    print('%s: %0.1f%% currently liked (of %d total tracks) ' % (pl, 100*len(tracks_db_liked)/len(track_df), len(track_df)))

playlist_liked_tracks = pd.concat(playlist_liked_tracks).sort_values('artist')
playlist_liked_tracks = playlist_liked_tracks.loc[~playlist_liked_tracks.index.duplicated(keep='first'), LIKE_TRACKS_HEADER] 
new_like_df = update_like_tsv(playlist_liked_tracks)


Found 47 like playlists out ot the 258 total
1950s_thumbs_up.tsv: 98.4% currently liked (of 61 total tracks) 
1960s_thumbs_up.tsv: 95.2% currently liked (of 1450 total tracks) 
1970s_thumbs_up.tsv: 81.4% currently liked (of 2143 total tracks) 
1980s_thumbs_up.tsv: 97.0% currently liked (of 533 total tracks) 
1990s_thumbs_up.tsv: 88.4% currently liked (of 1273 total tracks) 
2000s_thumbs_up.tsv: 93.3% currently liked (of 985 total tracks) 
2005s_thumbs_up.tsv: 87.6% currently liked (of 1417 total tracks) 
2010_thumbs_up.tsv: 91.8% currently liked (of 790 total tracks) 
2010s Electronic like.tsv: 100.0% currently liked (of 50 total tracks) 
2011_thumbs_up.tsv: 93.2% currently liked (of 807 total tracks) 
2012_thumbs_up.tsv: 92.4% currently liked (of 801 total tracks) 
2013_thumbs_up.tsv: 88.4% currently liked (of 1045 total tracks) 
2014_thumbs_up.tsv: 89.4% currently liked (of 792 total tracks) 
2015_thumbs_up.tsv: 94.5% currently liked (of 635 total tracks) 
2016_thumbs_up.tsv: 92.9% c